<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/03-oop-for-ml/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Exercises

Twelve classes to write: four Easy, five Medium and three Hard. Each exercise has a stub to fill in and a check cell.

The checks work a bit differently from earlier modules. Since you're writing classes, each check runs short scenarios, like "create two instances and make sure they don't share state" or "fit on one dataset, then transform another". If one fails, it tells you which scenario broke.

**Run the setup cell first.** Besides the checks, it defines a `Metric` base class you'll need in exercise 9, and some small helper classes used to test the `Pipeline` in exercise 10.

Rules:

- Standard library only.
- Keep the class names and method names exactly as described, or the checks can't find them.
- Solutions are in `03_solutions.ipynb`.

In [ ]:
# Setup: run this cell once before you start.
# It defines the check functions, plus a few helper classes that some
# exercises need (Metric, and the small transformers used to test Pipeline).

import copy
import math
from abc import ABC, abstractmethod


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- helpers used by the exercises ----

class Metric(ABC):
    """Base class for metrics that accumulate over batches (used in exercise 9)."""

    def __init__(self):
        self.reset()

    @abstractmethod
    def reset(self):
        """Clear all accumulated state."""

    @abstractmethod
    def update(self, y_true, y_pred):
        """Add one batch of labels and predictions."""

    @abstractmethod
    def compute(self):
        """Return the metric over everything seen since the last reset."""


class AddConstant:
    """Transformer: adds a constant to every value (used to test Pipeline)."""
    def __init__(self, c):
        self.c = c

    def fit(self, X):
        return self

    def transform(self, X):
        return [x + self.c for x in X]


class DivideByMax:
    """Transformer: divides by the largest value seen during fit."""
    def fit(self, X):
        self.max_ = max(X)
        return self

    def transform(self, X):
        return [x / self.max_ for x in X]


class Echo:
    """Estimator: 'predicts' its input unchanged, so tests can see what reached it."""
    def fit(self, X, y):
        return self

    def predict(self, X):
        return list(X)


class MeanRegressor:
    """Estimator: always predicts the mean of the training targets."""
    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)
        return self

    def predict(self, X):
        return [self.mean_ for _ in X]


# ---- checks ----

def check_RunningMean(C):
    def three_values():
        r = C()
        for v in (2, 4, 6):
            r.add(v)
        return (r.count, r.mean())

    def independent():
        r1, r2 = C(), C()
        r1.add(10)
        return (r1.mean(), r2.mean(), r2.count)

    _scenarios("RunningMean", [
        ("a new object has count 0", lambda: C().count, 0, "set self.count in __init__"),
        ("mean() with no values is 0.0", lambda: C().mean(), 0.0, None),
        ("add 2, 4, 6 then check (count, mean)", three_values, (3, 4.0), None),
        ("two objects don't share state", independent, (10.0, 0.0, 0),
         "create the running total and count in __init__, not in the class body"),
    ])


def check_LabelEncoder(C):
    def fit_returns_self():
        e = C()
        return e.fit(["b", "a"]) is e

    fitted = lambda: C().fit(["dog", "cat", "dog", "bird"])
    _scenarios("LabelEncoder", [
        ("fit() returns self", fit_returns_self, True, "end fit() with `return self`"),
        ("classes_ is the sorted unique labels", lambda: fitted().classes_, ["bird", "cat", "dog"], None),
        ("transform", lambda: fitted().transform(["cat", "dog", "cat"]), [1, 2, 1], None),
        ("inverse_transform", lambda: fitted().inverse_transform([0, 2]), ["bird", "dog"], None),
        ("unknown label", lambda: fitted().transform(["fish"]), Raises(ValueError),
         "raise ValueError for labels not seen during fit"),
    ])


def check_ExponentialMovingAverage(C):
    def sequence():
        e = C(0.5)
        return [e(10), e(20), e(20)]

    def value_after():
        e = C(0.5)
        e(4)
        e(8)
        return e.value

    _scenarios("ExponentialMovingAverage", [
        ("value is None before the first call", lambda: C(0.3).value, None, None),
        ("the first call returns the value itself", lambda: C(0.1)(5.0), 5.0,
         "there's no history yet, so the average is just the first value"),
        ("alpha=0.5 over 10, 20, 20", sequence, [10.0, 15.0, 17.5], "new = alpha * x + (1 - alpha) * old"),
        ("value attribute after two calls", value_after, 6.0, None),
        ("alpha=0 is invalid", lambda: C(0), Raises(ValueError), "alpha must be in (0, 1]"),
        ("alpha=1.5 is invalid", lambda: C(1.5), Raises(ValueError), None),
    ])


def check_Subset(C):
    base = ["a", "b", "c", "d", "e"]
    s = lambda: C(base, [4, 0, 2])
    _scenarios("Subset", [
        ("len", lambda: len(s()), 3, "implement __len__"),
        ("first item", lambda: s()[0], "e", "implement __getitem__: map position i to base[indices[i]]"),
        ("last item", lambda: s()[2], "c", None),
        ("negative index", lambda: s()[-1], "c", None),
        ("iterating with a for loop", lambda: [x for x in s()], ["e", "a", "c"],
         "a for loop calls s[0], s[1], ... until IndexError. Make sure an out-of-range index raises IndexError"),
    ])


def check_MinMaxScaler(C):
    def fit_returns_self():
        sc = C()
        return sc.fit([[1, 2], [3, 4]]) is sc

    def input_untouched():
        X = [[1, 2], [3, 4]]
        C().fit_transform(X)
        return X

    _scenarios("MinMaxScaler", [
        ("fit() returns self", fit_returns_self, True, None),
        ("fit_transform", lambda: C().fit_transform([[1, 10], [3, 30], [5, 50]]),
         [[0.0, 0.0], [0.5, 0.5], [1.0, 1.0]], "scale each column separately"),
        ("transform uses the min/max learned in fit",
         lambda: C().fit([[0, 100], [10, 200]]).transform([[5, 150], [20, 0]]),
         [[0.5, 0.5], [2.0, -1.0]], "don't refit or clip on new data: values outside the training range go outside 0..1"),
        ("constant column becomes 0.0", lambda: C().fit_transform([[1, 7], [3, 7]]),
         [[0.0, 0.0], [1.0, 0.0]], None),
        ("transform before fit", lambda: C().transform([[1]]), Raises(RuntimeError), None),
        ("input is not modified", input_untouched, [[1, 2], [3, 4]], "build new lists instead of changing X"),
    ])


def check_EarlyStopping(C):
    def basic():
        es = C(patience=2)
        return [es.step(loss) for loss in [1.0, 0.8, 0.85, 0.9, 0.7]]

    def with_delta():
        es = C(patience=1, min_delta=0.1)
        return [es.step(loss) for loss in [1.0, 0.95, 0.85]]

    def tracks_best():
        es = C()
        for loss in [0.5, 0.4, 0.6]:
            es.step(loss)
        return es.best

    def nan_loss():
        es = C(patience=1)
        return [es.step(1.0), es.step(float("nan")), es.best]

    _scenarios("EarlyStopping", [
        ("best starts at infinity", lambda: C().best, float("inf"), None),
        ("patience=2", basic, [False, False, False, True, False],
         "stop after `patience` steps in a row without improvement; an improvement resets the count"),
        ("min_delta=0.1", with_delta, [False, True, False], "an improvement means loss < best - min_delta"),
        ("best is the lowest loss so far", tracks_best, 0.4, None),
        ("nan is never an improvement", nan_loss, [False, True, 1.0],
         "any comparison with nan is False. Write the check so that False means 'no improvement'"),
    ])


def check_Vector(C):
    def copies_input():
        data = [1, 2]
        v = C(data)
        data.append(3)
        return len(v)

    def unchanged():
        a = C([1, 2])
        a * 3
        a + C([1, 1])
        return a.values

    _scenarios("Vector", [
        ("repr", lambda: repr(C([1, 2, 3])), "Vector([1, 2, 3])", None),
        ("len", lambda: len(C([1, 2, 3])), 3, None),
        ("indexing", lambda: C([7, 8, 9])[1], 8, None),
        ("addition", lambda: (C([1, 2]) + C([3, 4])).values, [4, 6], None),
        ("subtraction", lambda: (C([5, 5]) - C([1, 2])).values, [4, 3], None),
        ("vector * number", lambda: (C([1, 2]) * 3).values, [3, 6], None),
        ("number * vector", lambda: (3 * C([1, 2])).values, [3, 6], "implement __rmul__"),
        ("dot product", lambda: C([1, 2, 3]).dot(C([4, 5, 6])), 32, None),
        ("equal vectors", lambda: C([1, 2]) == C([1, 2]), True, None),
        ("different vectors", lambda: C([1, 2]) == C([2, 1]), False, None),
        ("comparing with a non-Vector", lambda: C([1, 2]) == [1, 2], False, "return NotImplemented for other types"),
        ("adding vectors of different lengths", lambda: C([1, 2]) + C([1, 2, 3]), Raises(ValueError), None),
        ("operations don't modify the original", unchanged, [1, 2], "return a new Vector"),
        ("the constructor copies the list", copies_input, 2, "store list(values), not the caller's list itself"),
    ])


def check_TrainConfig(C):
    def defaults():
        c = C()
        return (c.lr, c.batch_size, c.epochs, c.hidden_sizes, c.dropout)

    def separate_lists():
        a, b = C(), C()
        a.hidden_sizes.append(16)
        return b.hidden_sizes

    _scenarios("TrainConfig", [
        ("default values", defaults, (0.001, 32, 10, [64, 32], 0.0), None),
        ("keyword arguments", lambda: C(lr=0.01, epochs=3).epochs, 3, None),
        ("instances don't share hidden_sizes", separate_lists, [64, 32], "use field(default_factory=...)"),
        ("equal configs compare equal", lambda: C(lr=0.01) == C(lr=0.01), True, "use @dataclass"),
        ("lr=0 is invalid", lambda: C(lr=0), Raises(ValueError), "validate in __post_init__"),
        ("batch_size=0 is invalid", lambda: C(batch_size=0), Raises(ValueError), None),
        ("dropout=1.0 is invalid", lambda: C(dropout=1.0), Raises(ValueError), "dropout must be in [0, 1)"),
        ("num_steps", lambda: C(batch_size=32, epochs=2).num_steps(100), 8, "100 examples in batches of 32 is 4 batches (round up)"),
    ])


def check_Accuracy(C):
    def one_batch():
        m = C()
        m.update([1, 0, 1], [1, 1, 1])
        return m.compute()

    def two_batches():
        m = C()
        m.update([1, 1], [1, 1])
        m.update([0, 0, 0, 0], [1, 1, 1, 1])
        return m.compute()

    def after_reset():
        m = C()
        m.update([1], [1])
        m.reset()
        return m.compute()

    _scenarios("Accuracy", [
        ("is a Metric", lambda: isinstance(C(), Metric), True, "class Accuracy(Metric):"),
        ("one batch", one_batch, 2 / 3, None),
        ("two batches of different sizes", two_batches, 2 / 6,
         "count correct predictions over all examples, don't average the batch accuracies"),
        ("compute() with no data", lambda: C().compute(), 0.0, None),
        ("reset() clears everything", after_reset, 0.0, None),
        ("mismatched lengths", lambda: C().update([1, 0], [1]), Raises(ValueError), None),
    ])


def check_Pipeline(C):
    def build():
        return C([("add", AddConstant(1)), ("div", DivideByMax()), ("out", Echo())])

    def fit_returns_self():
        p = build()
        return p.fit([1, 3], [0, 0]) is p

    _scenarios("Pipeline", [
        ("fit() returns self", fit_returns_self, True, None),
        ("each step is fitted on the previous step's output", lambda: build().fit([1, 3], [0, 0])["div"].max_, 4,
         "DivideByMax should see [2, 4] (after adding 1), so its max_ is 4"),
        ("predict applies the steps in order", lambda: build().fit([1, 3], [0, 0]).predict([3, 7]), [1.0, 2.0],
         "transform X with each step in order, then call predict on the last step"),
        ("access a step by name", lambda: isinstance(build()["add"], AddConstant), True, "implement __getitem__"),
        ("len", lambda: len(build()), 3, None),
        ("the final step receives y",
         lambda: C([("add", AddConstant(0)), ("m", MeanRegressor())]).fit([1, 2], [10, 20]).predict([5, 5, 5]),
         [15.0, 15.0, 15.0], None),
        ("duplicate step names", lambda: C([("a", AddConstant(1)), ("a", Echo())]), Raises(ValueError), None),
    ])


def check_WarmupCosineSchedule(C):
    s = lambda: C(1.0, warmup_steps=4, total_steps=14)
    _scenarios("WarmupCosineSchedule", [
        ("step 0 (warmup)", lambda: s()(0), 0.25, "during warmup: base_lr * (step + 1) / warmup_steps"),
        ("last warmup step", lambda: s()(3), 1.0, None),
        ("first step after warmup", lambda: s()(4), 1.0, None),
        ("halfway through the decay", lambda: s()(9), 0.5, None),
        ("final step", lambda: s()(14), 0.0, None),
        ("past the end stays at min_lr", lambda: s()(100), 0.0, "clamp progress to at most 1.0"),
        ("min_lr, halfway", lambda: C(1.0, warmup_steps=0, total_steps=10, min_lr=0.1)(5), 0.55, None),
        ("min_lr, at the end", lambda: C(1.0, warmup_steps=0, total_steps=10, min_lr=0.1)(10), 0.1, None),
        ("warmup_steps >= total_steps", lambda: C(1.0, warmup_steps=10, total_steps=10), Raises(ValueError), None),
    ])


def check_ConfusionMatrix(C):
    labels = ["cat", "dog", "bird"]
    y_true = ["cat", "cat", "dog", "bird", "dog", "cat"]
    y_pred = ["cat", "dog", "dog", "bird", "cat", "cat"]

    def filled():
        cm = C(labels)
        cm.update(y_true[:3], y_pred[:3])
        cm.update(y_true[3:], y_pred[3:])
        return cm

    def safe_matrix():
        cm = filled()
        m = cm.matrix
        m[0][0] = 99
        return cm[("cat", "cat")]

    def zero_division():
        cm = C(labels)
        cm.update(["cat"], ["cat"])
        return (cm.precision("dog"), cm.recall("dog"))

    _scenarios("ConfusionMatrix", [
        ("matrix (rows are true labels, columns are predictions)", lambda: filled().matrix,
         [[2, 1, 0], [1, 1, 0], [0, 0, 1]], "counts should add up over several update() calls. If a whole column changes at once, check how you created the rows"),
        ("cm[(true, pred)]", lambda: filled()[("cat", "dog")], 1, None),
        ("accuracy", lambda: filled().accuracy(), 4 / 6, None),
        ("precision('cat')", lambda: filled().precision("cat"), 2 / 3, "of everything predicted 'cat', how much really was 'cat'"),
        ("recall('dog')", lambda: filled().recall("dog"), 1 / 2, "of everything that really was 'dog', how much was predicted 'dog'"),
        ("precision('bird')", lambda: filled().precision("bird"), 1.0, None),
        ("no predictions or examples of a class", zero_division, (0.0, 0.0), "return 0.0 instead of dividing by zero"),
        ("changing .matrix doesn't change the counts", safe_matrix, 2, "return a copy of the rows from the matrix property"),
        ("unknown label", lambda: C(labels).update(["fish"], ["cat"]), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

## Easy

### Exercise 1: `RunningMean`

Write a class `RunningMean` that keeps track of the mean of a stream of numbers.

- `add(x)` adds a value.
- `mean()` returns the mean of all values added so far, or `0.0` if none have been added.
- `count` is an attribute holding how many values have been added.

```python
r = RunningMean()
r.add(2); r.add(4); r.add(6)
r.count    # 3
r.mean()   # 4.0
```

Don't store every value. A running total and a count are enough. This is how training loops track the average loss over an epoch without keeping a list of thousands of numbers.

In [ ]:
class RunningMean:
    # your code here
    pass

In [ ]:
check_RunningMean(RunningMean)

### Exercise 2: `LabelEncoder`

Write a scikit-learn-style `LabelEncoder`.

- `fit(labels)` learns the classes and stores them in `self.classes_` as a sorted list of the unique labels. It returns `self`.
- `transform(labels)` returns the list of integer ids, where each id is the label's position in `classes_`. Raise a `ValueError` for a label not seen during `fit`.
- `inverse_transform(ids)` turns ids back into labels.

```python
enc = LabelEncoder().fit(["dog", "cat", "dog", "bird"])
enc.classes_                       # ["bird", "cat", "dog"]
enc.transform(["cat", "dog"])      # [1, 2]
enc.inverse_transform([0, 2])      # ["bird", "dog"]
```

In [ ]:
class LabelEncoder:
    # your code here
    pass

In [ ]:
check_LabelEncoder(LabelEncoder)

### Exercise 3: `ExponentialMovingAverage`

An exponential moving average (EMA) smooths a noisy series by giving recent values more weight. It's how TensorBoard smooths loss curves, and some models keep an EMA of their weights for more stable results.

Write a class `ExponentialMovingAverage(alpha)`:

- `alpha` must be greater than 0 and at most 1. Otherwise raise a `ValueError`.
- Calling the object with a value, as in `ema(x)`, updates the average and returns the new value. The first call just returns `x`. After that, `new = alpha * x + (1 - alpha) * old`.
- `value` is an attribute holding the current average, or `None` before the first call.

```python
ema = ExponentialMovingAverage(0.5)
ema(10)   # 10.0
ema(20)   # 15.0
ema(20)   # 17.5
```

In [ ]:
class ExponentialMovingAverage:
    # your code here
    pass

In [ ]:
check_ExponentialMovingAverage(ExponentialMovingAverage)

### Exercise 4: `Subset`

PyTorch's `Subset` wraps a dataset and exposes only some of its examples, without copying anything. It's used to split one dataset into train and validation parts.

Write `Subset(dataset, indices)`:

- `len(subset)` is the number of indices.
- `subset[i]` returns `dataset[indices[i]]`.

`dataset` can be anything that supports `[]`, like a list or the `TabularDataset` from the lesson.

```python
s = Subset(["a", "b", "c", "d", "e"], [4, 0, 2])
len(s)    # 3
s[0]      # "e"
list(s)   # ["e", "a", "c"]
```

In [ ]:
class Subset:
    # your code here
    pass

In [ ]:
check_Subset(Subset)

## Medium

### Exercise 5: `MinMaxScaler`

Write a `MinMaxScaler` that works on 2D data: a list of rows, where each row is a list of feature values. Each column (feature) is scaled separately.

- `fit(X)` learns the min and max of each column and returns `self`.
- `transform(X)` scales every value to `(x - min) / (max - min)` using the numbers learned in `fit`. If a column has `max == min`, output `0.0` for it. Calling `transform` before `fit` raises a `RuntimeError`.
- `fit_transform(X)` fits and transforms in one step.
- Don't modify `X`.

```python
MinMaxScaler().fit_transform([[1, 10], [3, 30], [5, 50]])
# [[0.0, 0.0], [0.5, 0.5], [1.0, 1.0]]
```

Values in new data can fall outside 0 to 1. Don't clip them.

In [ ]:
class MinMaxScaler:
    # your code here
    pass

In [ ]:
check_MinMaxScaler(MinMaxScaler)

### Exercise 6: `EarlyStopping`

Write an `EarlyStopping(patience=3, min_delta=0.0)` class that decides when to stop training.

- `best` is an attribute holding the lowest loss seen so far. It starts at `float("inf")`.
- `step(loss)` is called once per epoch with the validation loss, and returns `True` if training should stop.
- A loss counts as an improvement only if `loss < best - min_delta`. An improvement updates `best` and resets the count of bad epochs.
- Any other loss adds one to the count of bad epochs. When that count reaches `patience`, `step` returns `True`.

```python
es = EarlyStopping(patience=2)
[es.step(loss) for loss in [1.0, 0.8, 0.85, 0.9]]
# [False, False, False, True]
```

Think about what should happen if the loss is `nan`.

In [ ]:
class EarlyStopping:
    # your code here
    pass

In [ ]:
check_EarlyStopping(EarlyStopping)

### Exercise 7: `Vector`

Write a `Vector` class for vectors of any length.

- `Vector(values)` stores a **copy** of the list in `self.values`.
- `repr(v)` looks like `Vector([1, 2, 3])`.
- `len(v)` and `v[i]` work.
- `v + w` and `v - w` work element by element and return a new `Vector`. Raise a `ValueError` if the lengths differ.
- `v * 3` and `3 * v` multiply every element by a number.
- `v.dot(w)` returns the dot product: the sum of the products of matching elements.
- `v == w` is `True` if the values are equal. Comparing with something that isn't a `Vector` gives `False`.

None of these should modify the original vectors.

In [ ]:
class Vector:
    # your code here
    pass

In [ ]:
check_Vector(Vector)

### Exercise 8: `TrainConfig`

Write a `TrainConfig` **dataclass** with these fields and defaults:

| field | default |
|---|---|
| `lr` | `1e-3` |
| `batch_size` | `32` |
| `epochs` | `10` |
| `hidden_sizes` | `[64, 32]` |
| `dropout` | `0.0` |

- Each instance must get its own `hidden_sizes` list.
- Validate in `__post_init__`: raise a `ValueError` if `lr <= 0`, if `batch_size < 1`, or if `dropout` isn't in the range `[0, 1)`.
- Add a method `num_steps(n_examples)` that returns the total number of optimizer steps for the whole run: the number of batches per epoch (rounded up) times `epochs`.

```python
TrainConfig(batch_size=32, epochs=2).num_steps(100)   # 8
```

In [ ]:
from dataclasses import dataclass, field


class TrainConfig:
    # your code here
    pass

In [ ]:
check_TrainConfig(TrainConfig)

### Exercise 9: `Accuracy`

Metrics in libraries like `torchmetrics` are classes that accumulate results over many batches and compute the final number at the end. The setup cell defines this base class:

```python
class Metric(ABC):
    def __init__(self):
        self.reset()

    @abstractmethod
    def reset(self): ...

    @abstractmethod
    def update(self, y_true, y_pred): ...

    @abstractmethod
    def compute(self): ...
```

Write `Accuracy` as a subclass of `Metric`:

- `update(y_true, y_pred)` adds one batch. Raise a `ValueError` if the lengths differ.
- `compute()` returns the accuracy over **every example** seen since the last reset, or `0.0` if there are none.
- `reset()` clears everything.

Careful: the accuracy over two batches is not the average of the two batch accuracies, unless the batches are the same size.

In [ ]:
class Accuracy(Metric):
    # your code here
    pass

In [ ]:
check_Accuracy(Accuracy)

## Hard

### Exercise 10: `Pipeline`

Write a scikit-learn-style `Pipeline` that chains preprocessing steps and a final model.

`Pipeline(steps)` takes a list of `(name, object)` pairs. Every step except the last is a *transformer* with `fit(X)` and `transform(X)` methods. The last step is an *estimator* with `fit(X, y)` and `predict(X)` methods.

- `fit(X, y)`: fit each transformer on the data, then transform the data with it and pass the result to the next step. Finally, fit the estimator on the fully transformed data and `y`. Return `self`.
- `predict(X)`: transform `X` through every transformer (without fitting them again), then return the estimator's predictions.
- `pipe[name]` returns the step object with that name.
- `len(pipe)` is the number of steps.
- Raise a `ValueError` in the constructor if two steps have the same name.

The setup cell defines some small classes the checks use, like `AddConstant(c)`, `DivideByMax()` and `MeanRegressor()`. You can use them to try your pipeline out.

In [ ]:
class Pipeline:
    # your code here
    pass

In [ ]:
check_Pipeline(Pipeline)

### Exercise 11: `WarmupCosineSchedule`

Most modern models, including transformers, are trained with a learning rate that changes over time: it rises linearly during a short *warmup*, then decays along a cosine curve. Write `WarmupCosineSchedule(base_lr, warmup_steps, total_steps, min_lr=0.0)`. Calling it as `schedule(step)` returns the learning rate for that step:

- If `step < warmup_steps`: `base_lr * (step + 1) / warmup_steps`
- Otherwise, let `progress = (step - warmup_steps) / (total_steps - warmup_steps)`, capped at `1.0`. Then:

$$\text{lr} = \text{min\_lr} + \frac{1}{2}(\text{base\_lr} - \text{min\_lr})\left(1 + \cos(\pi \cdot \text{progress})\right)$$

Raise a `ValueError` if `warmup_steps >= total_steps`. `warmup_steps` can be 0.

```python
s = WarmupCosineSchedule(1.0, warmup_steps=4, total_steps=14)
[round(s(i), 3) for i in [0, 3, 4, 9, 14]]   # [0.25, 1.0, 1.0, 0.5, 0.0]
```

Import `math` for `math.cos` and `math.pi`.

In [ ]:
import math


class WarmupCosineSchedule:
    # your code here
    pass

In [ ]:
check_WarmupCosineSchedule(WarmupCosineSchedule)

### Exercise 12: `ConfusionMatrix`

In Module 1 you counted true and false positives for two classes. Now generalise it to any number of classes. Write `ConfusionMatrix(labels)`, where `labels` is a list of class names. The order of `labels` sets the order of the rows and columns.

- `update(y_true, y_pred)` adds a batch of results. It can be called many times, and counts add up. Raise a `ValueError` for any label not in `labels`.
- `matrix` is a **property** returning a list of lists, where `matrix[i][j]` is how many examples had true label `labels[i]` and were predicted as `labels[j]`. Changing the returned lists must not change the stored counts.
- `cm[(true_label, pred_label)]` returns a single count.
- `accuracy()` returns the fraction of all examples on the diagonal (predicted correctly).
- `precision(label)`: of everything predicted as `label`, the fraction that really was `label`.
- `recall(label)`: of everything that really was `label`, the fraction predicted as `label`.
- `precision` and `recall` return `0.0` when they would divide by zero.

```python
cm = ConfusionMatrix(["cat", "dog"])
cm.update(["cat", "cat", "dog"], ["cat", "dog", "dog"])
cm.matrix              # [[1, 1], [0, 1]]
cm[("cat", "dog")]     # 1
```

In [ ]:
class ConfusionMatrix:
    # your code here
    pass

In [ ]:
check_ConfusionMatrix(ConfusionMatrix)

## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then head to the mini project in `04_mini_project.ipynb`, where you'll use operator overloading to build a small automatic differentiation engine and train a neural network with it.